# 👨‍🍳 Azure Translator Recipes

## Creating the client: `TextTranslationClient`

In [ ]:
import os
from uuid import uuid4

from azure.ai.translation.text import TextTranslationClient
from azure.core.credentials import AzureKeyCredential

key = os.environ["AZURE_TRANSLATOR_KEY"]
region = os.environ["AZURE_TRANSLATOR_REGION"]

client = TextTranslationClient(
    endpoint="https://api.cognitive.microsofttranslator.com",  # global; api-eur, api-nam or api-apc pin a geography
    credential=AzureKeyCredential(key),                        # resource key, or a token credential from azure-identity
    region=region,                                             # resource region; required with a key on this endpoint
    resource_id=None,                                          # resource ID; required with a token credential on this endpoint
    audience=None,                                             # token audience override; unused with a key
    api_version="2026-06-06",                                  # service API version (the SDK 2.0.0 default)
)

## Getting supported languages: `get_supported_languages`

In [ ]:
from azure.core import MatchConditions

result = client.get_supported_languages(
    client_trace_id=str(uuid4()),                     # client-generated GUID for tracing the request
    scope="translation,transliteration,models",       # language groups to return (default: all three)
    accept_language="en",                             # locale for display names (default: en)
    etag=None,                                        # ETag of an earlier response, for a conditional request
    match_condition=MatchConditions.Unconditionally,  # how to apply etag; Unconditionally ignores it
)

translation_languages = result.translation or {}
transliteration_languages = result.transliteration or {}
print("Models:", result.models)
print("Translation languages:", len(translation_languages))
print("Transliteration languages:", len(transliteration_languages))

for code in ["ar", "ja"]:  # two languages that support both translation and transliteration
    language = translation_languages[code]
    print("Translation language code:", code)
    print("Name:", language.name)
    print("Native name:", language.native_name)
    print("Directionality:", language.dir)
    print("Models:", language.models)

for code in ["ar", "ja"]:
    language = transliteration_languages[code]
    print("Transliteration language code:", code)
    print("Name:", language.name)
    print("Native name:", language.native_name)
    for script in language.scripts:
        print("Source script code:", script.code)
        print("Source script name:", script.name)
        print("Source script native name:", script.native_name)
        print("Source script directionality:", script.dir)
        for target_script in script.to_scripts:
            print("Target script code:", target_script.code)
            print("Target script name:", target_script.name)
            print("Target script native name:", target_script.native_name)
            print("Target script directionality:", target_script.dir)

## Sending a custom request: `send_request`

In [ ]:
from azure.core.rest import HttpRequest

body = {
    "inputs": [
        {"text": "Where is the nearest train station?", "language": "en", "targets": [{"language": "it"}]},
    ],
}

request = HttpRequest(
    method="POST",                              # HTTP method of the operation
    url="/translate",                           # relative URLs resolve against the client's endpoint
    params={"api-version": "2026-06-06"},       # query string parameters
    headers={"X-ClientTraceId": str(uuid4())},  # extra request headers
    json=body,                                  # sent as the JSON request body
)

result = client.send_request(
    request=request,  # sent through the client's authentication, retry and logging policies
    stream=False,     # True = leave the response body unread so you can stream it
)
result.raise_for_status()

print("Status code:", result.status_code)
for name, value in result.headers.items():
    print(f"Header {name}:", value)
for item in result.json()["value"]:
    for translation in item["translations"]:
        print("Target language:", translation["language"])
        print("Translated text:", translation["text"])

## Translating text: `translate`

In [ ]:
texts = [
    "The museum opens at nine in the morning.",
    "Tickets are free on the first Sunday of every month.",
]

result = client.translate(
    body=texts,                       # strings, or TranslateInputItem objects (see the next recipes)
    to_language=["fr", "ja"],         # target languages for every string
    from_language="en",               # source language; None = detect it for each string
    client_trace_id=str(uuid4()),     # client-generated GUID for tracing the request
    content_type="application/json",  # media type of the request body
)

for item in result:
    if item.detected_language:
        print("Detected language:", item.detected_language.language)
        print("Detection score:", item.detected_language.score)
    for translation in item.translations:
        print("Target language:", translation.language)
        print("Translated text:", translation.text)
        print("Source characters:", translation.source_characters)

## Translating with input and target options: `translate`

In [ ]:
from azure.ai.translation.text.models import (
    ProfanityAction,
    ProfanityMarker,
    TextType,
    TranslateInputItem,
    TranslationTarget,
)

html = '<p>Book a table at <span class="notranslate">The Blue Door</span> tonight.</p>'

inputs = [
    TranslateInputItem(
        text=html,                                   # notranslate elements are left untranslated
        targets=[TranslationTarget(language="de")],  # one translation per target
        language="en",                               # source language; None = detect it
        text_type=TextType.HTML,                     # Plain (default) or Html
    ),
    TranslateInputItem(
        text="hudha akhtabar.",                                          # Arabic written in Latin script
        targets=[TranslationTarget(language="zh-Hans", script="Latn")],  # translate, then transliterate
        script="Latn",                                                   # script of the input text
        language="ar",                                                   # source language
    ),
    TranslateInputItem(
        text="That was a damn good match.",  # no language, so the service detects it
        targets=[
            TranslationTarget(
                language="es",                            # target language
                profanity_action=ProfanityAction.MARKED,  # NoAction (default), Marked or Deleted
                profanity_marker=ProfanityMarker.TAG,     # Asterisk (default) or Tag
            ),
        ],
    ),
]

result = client.translate(
    body=inputs,                      # TranslateInputItem objects carry their own source and targets
    to_language=None,                 # only applies to string inputs
    from_language=None,               # only applies to string inputs
    client_trace_id=str(uuid4()),     # client-generated GUID for tracing the request
    content_type="application/json",  # media type of the request body
)

for item in result:
    if item.detected_language:
        print("Detected language:", item.detected_language.language)
        print("Detection score:", item.detected_language.score)
    for translation in item.translations:
        print("Target language:", translation.language)
        print("Translated text:", translation.text)
        print("Source characters:", translation.source_characters)

## Translating with a custom model: `translate`

In [ ]:
# Requires a custom translation model published with this Translator resource.
from azure.ai.translation.text.models import TranslateInputItem, TranslationTarget

inputs = [
    TranslateInputItem(
        text="Replace the brake pads when they are thinner than three millimetres.",
        targets=[
            TranslationTarget(
                language="de",                         # a target language your model was trained on
                deployment_name="<your-category-id>",  # category ID of your published model
                allow_fallback=False,                  # False = fail instead of using the general model
            ),
        ],
        language="en",
    ),
]

result = client.translate(
    body=inputs,                      # TranslateInputItem objects carry their own source and targets
    to_language=None,                 # only applies to string inputs
    from_language=None,               # only applies to string inputs
    client_trace_id=str(uuid4()),     # client-generated GUID for tracing the request
    content_type="application/json",  # media type of the request body
)

for item in result:
    if item.detected_language:
        print("Detected language:", item.detected_language.language)
        print("Detection score:", item.detected_language.score)
    for translation in item.translations:
        print("Target language:", translation.language)
        print("Translated text:", translation.text)
        print("Source characters:", translation.source_characters)

## Translating with an LLM deployment: `translate`

In [ ]:
# Requires a client created with the key and region of a Microsoft Foundry resource
# that has a deployment of a model supported for translation.
from azure.ai.translation.text.models import (
    TranslateInputItem,
    TranslationGender,
    TranslationTarget,
    TranslationTone,
)

inputs = [
    TranslateInputItem(
        text="The doctor is available next Monday. Would you like to book an appointment?",
        targets=[
            TranslationTarget(
                language="es",                                 # target language
                deployment_name="<your-llm-deployment-name>",  # your Foundry model deployment
                allow_fallback=True,                           # True (default) = use a general model if the LLM can't translate
                tone=TranslationTone.FORMAL,                   # formal, informal or neutral
                gender=TranslationGender.FEMALE,               # female, male or neutral
            ),
            TranslationTarget(
                language="es",
                deployment_name="<your-llm-deployment-name>",
                allow_fallback=True,
                tone=TranslationTone.INFORMAL,
                gender=TranslationGender.MALE,
            ),
        ],
        language="en",
    ),
]

result = client.translate(
    body=inputs,                      # TranslateInputItem objects carry their own source and targets
    to_language=None,                 # only applies to string inputs
    from_language=None,               # only applies to string inputs
    client_trace_id=str(uuid4()),     # client-generated GUID for tracing the request
    content_type="application/json",  # media type of the request body
)

for item in result:
    if item.detected_language:
        print("Detected language:", item.detected_language.language)
        print("Detection score:", item.detected_language.score)
    for translation in item.translations:
        print("Target language:", translation.language)
        print("Translated text:", translation.text)
        print("Source characters:", translation.source_characters)
        print("Instruction tokens:", translation.instruction_tokens)
        print("Source tokens:", translation.source_tokens)
        print("Response tokens:", translation.response_tokens)
        print("Target tokens:", translation.target_tokens)

## Adapting LLM translations to your domain: `translate`

In [ ]:
# Requires a client created with the key and region of a Microsoft Foundry resource
# that has a deployment of a model supported for translation.
from azure.ai.translation.text.models import ReferenceTextPair, TranslateInputItem, TranslationTarget

inputs = [
    TranslateInputItem(
        text="Restart the gateway before you update its firmware.",
        targets=[
            TranslationTarget(
                language="fr",                                 # target language
                deployment_name="<your-llm-deployment-name>",  # your Foundry model deployment
                reference_text_pairs=[                         # up to five example translations; overrides adaptive_dataset_id
                    ReferenceTextPair(
                        source="Restart the gateway.",       # example source text
                        target="Redémarrez la passerelle.",  # its approved translation
                    ),
                    ReferenceTextPair(
                        source="Update the firmware.",
                        target="Mettez à jour le micrologiciel.",
                    ),
                ],
            ),
            TranslationTarget(
                language="fr",
                deployment_name="<your-llm-deployment-name>",
                adaptive_dataset_id="<your-adaptive-dataset-id>",  # index of your domain sentence pairs
            ),
        ],
        language="en",
    ),
]

result = client.translate(
    body=inputs,                      # TranslateInputItem objects carry their own source and targets
    to_language=None,                 # only applies to string inputs
    from_language=None,               # only applies to string inputs
    client_trace_id=str(uuid4()),     # client-generated GUID for tracing the request
    content_type="application/json",  # media type of the request body
)

for item in result:
    if item.detected_language:
        print("Detected language:", item.detected_language.language)
        print("Detection score:", item.detected_language.score)
    for translation in item.translations:
        print("Target language:", translation.language)
        print("Translated text:", translation.text)
        print("Source characters:", translation.source_characters)
        print("Instruction tokens:", translation.instruction_tokens)
        print("Source tokens:", translation.source_tokens)
        print("Response tokens:", translation.response_tokens)
        print("Target tokens:", translation.target_tokens)

## Transliterating text: `transliterate`

In [ ]:
from azure.ai.translation.text.models import InputTextItem

inputs = [InputTextItem(text="这是个测试。")]

result = client.transliterate(
    body=inputs,                      # InputTextItem objects, or plain strings
    language="zh-Hans",               # language of the input text
    from_script="Hans",               # script of the input text
    to_script="Latn",                 # script to convert to
    client_trace_id=str(uuid4()),     # client-generated GUID for tracing the request
    content_type="application/json",  # media type of the request body
)

for item in result:
    print("Text:", item.text)
    print("Script:", item.script)

## Closing the client: `close`

In [ ]:
client.close()  # takes no parameters; closes the client's HTTP session